In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import classification_report
import time
import warnings
warnings.filterwarnings('ignore')

# Seed — final raporda zorunlu, her çalıştırmada aynı sonuç
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz     : {device}")
print(f"GPU       : {torch.cuda.get_device_name(0)}")
print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch   : {torch.__version__}")

Cihaz     : cuda
GPU       : Tesla T4
VRAM      : 15.6 GB
PyTorch   : 2.10.0+cu128


In [5]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/bdm_proje/data/raw/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 134
-rw------- 1 root root 137196 May 25 11:33 data.csv


In [4]:
df = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/raw/data.csv', sep=";")

After mounting your Google Drive, you'll need to specify the correct path to your `data.csv` file. You can typically find it under `/content/drive/MyDrive/` followed by the folder structure where you saved it. For example, if it's directly in your 'MyDrive', the path would be `'/content/drive/MyDrive/data.csv'`.

In [6]:
# Update this path to the correct location of your data.csv file in Google Drive
# For example, if it's in a folder named 'turkish-slm-benchmark' within your Drive, it might be:
# data_path = '/content/drive/MyDrive/turkish-slm-benchmark/data/raw/data.csv'
# To find the exact path, you can use `!ls /content/drive/MyDrive/` in a new cell.
data_path = '/content/drive/MyDrive/bdm_proje/data/raw/data.csv' # <<< Change this line to your actual path

df = pd.read_csv(data_path, sep=";")

In [7]:
df.head(20)

,tweet,label_id
0,Kahramanmaraş türkoğlu ilçesi şekeroba köyü ça...,0
1,"Teyitli, ses var, köpekler tepki veriyor. Her...",0
2,0539 693 27 99 bu arkadaş Kahramanmaraş’ta çad...,0
3,Babamın yaşadığı yere henüz yardım ulaşmamış ş...,0
4,Samsun Atakum'da 18 adet yeni eşyalı daire var...,3
5,Adıyaman merkez Bahçecik mahallesi yeni mezarl...,0
6,YETERSİZMİŞ ACİL YARDIM GİTMESİ LAZIM ( TEYİTL...,0
7,ACİL VİNÇ VE KEPÇE LAZIM SES VAR ‼️ SÜMERLER ...,0
8,TEYİTLİ 6 AYLIK HAMİLE CANSU KAVLAK. VİNÇ LAZ...,0
9,‼️ARKADAŞLAR YAYAR MISINIZ ‼️ SES VAR AMA SİS...,0


In [8]:
from sklearn.model_selection import train_test_split

LABEL_NAMES = {
    0: "Yardım Talebi",
    1: "Kayıp Bildirimi",
    2: "Altyapı Hasarı",
    3: "Bağış/Koordinasyon",
    4: "Diğer/İlgisiz"
}

print(f"Toplam veri : {len(df)} tweet")
print(f"\nEtiket dağılımı:")
for k, v in LABEL_NAMES.items():
    n = (df['label_id'] == k).sum()
    print(f"  {k} - {v:<22}: {n}")

# ADIM 1: %10 test ayır → geriye %90 kalır
train_val, test_df = train_test_split(
    df,
    test_size   = 0.10,        # %10 test
    random_state= SEED,
    stratify    = df['label_id']
)

# ADIM 2: kalan %90'ın %11.1'i → toplam %10 val olur
# Neden 0.111? → 0.90 * 0.111 ≈ 0.10
train_df, val_df = train_test_split(
    train_val,
    test_size   = 0.111,       # %90'ın %11.1'i = toplam %10 val
    random_state= SEED,
    stratify    = train_val['label_id']
)

print(f"\nBölme tamamlandı:")
print(f"  Train : {len(train_df)}  (%{len(train_df)/len(df)*100:.0f})")
print(f"  Val   : {len(val_df)}   (%{len(val_df)/len(df)*100:.0f})")
print(f"  Test  : {len(test_df)}   (%{len(test_df)/len(df)*100:.0f})")

print(f"\nTest seti etiket dağılımı:")
print(test_df['label_id'].value_counts().sort_index())

# Kaydet
train_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/val.csv',   index=False)
test_df.to_csv('/content/drive/MyDrive/bdm_proje/data/processed/test.csv',  index=False)

print("\n✅ Kaydedildi!")

Toplam veri : 793 tweet

Etiket dağılımı:
  0 - Yardım Talebi         : 289
  1 - Kayıp Bildirimi       : 98
  2 - Altyapı Hasarı        : 100
  3 - Bağış/Koordinasyon    : 105
  4 - Diğer/İlgisiz         : 201

Bölme tamamlandı:
  Train : 633  (%80)
  Val   : 80   (%10)
  Test  : 80   (%10)

Test seti etiket dağılımı:
label_id
0    29
1    10
2    10
3    11
4    20
Name: count, dtype: int64

✅ Kaydedildi!


In [9]:
#İLK DEFA MODELİ İNDİRİRKEN KULLANILACAK KOD


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Model yükleniyor: {MODEL_ID}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,     # torch_dtype yerine dtype
    device_map="auto"        # GPU varsa otomatik kullan
)

model.eval()

# Model bilgileri
param_count = sum(p.numel() for p in model.parameters())

model_size_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1e6

print("✅ Model yüklendi!")
print(f"Parametre sayısı : {param_count/1e6:.0f}M")
print(f"Model boyutu     : {model_size_mb:.0f} MB")

if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM kullanımı   : {torch.cuda.memory_allocated()/1e9:.2f} GB")




Model yükleniyor: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model yüklendi!
Parametre sayısı : 1100M
Model boyutu     : 2200 MB
GPU              : Tesla T4
VRAM kullanımı   : 2.20 GB


In [10]:
# İNDİRİLEN MODELİ KAYDETME KISMI

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Ensure these paths match your desired save location in Google Drive
tokenizer.save_pretrained("/content/drive/MyDrive/bdm_proje/models/tinyllama")
model.save_pretrained("/content/drive/MyDrive/bdm_proje/models/tinyllama")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Drive kopyası bozuk → doğrudan HuggingFace'ten yükle (~1-2 dk)
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,      # torch_dtype deprecated → dtype
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

model.eval()
print("✅ Model yüklendi (HuggingFace)")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"📊 VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model yüklendi (HuggingFace)
🚀 GPU: Tesla T4
📊 VRAM: 2.20 GB


In [14]:
import time

# Zero-shot ve fine-tuned AYNI prompt'u kullansın → adil karşılaştırma
def build_prompt(tweet):
    sistem = ("Sen bir deprem acil durum sınıflandırma asistanısın. "
              "Verilen tweeti aşağıdaki 5 kategoriden birine koy. "
              "SADECE kategori adını yaz, başka hiçbir şey yazma.")
    kullanici = (f"Tweet: {tweet}\n\n"
                 "Kategoriler:\n"
                 "- Yardım Talebi\n- Kayıp Bildirimi\n- Altyapı Hasarı\n"
                 "- Bağış/Koordinasyon\n- Diğer/İlgisiz\n\n"
                 "Kategori:")
    return (f"<|system|>\n{sistem}</s>\n"
            f"<|user|>\n{kullanici}</s>\n"
            f"<|assistant|>\n")

def parse_label(text):
    t = text.lower()
    if "yardım" in t or "yardim" in t:                    return 0
    if "kayıp" in t or "kayip" in t:                      return 1
    if "altyapı" in t or "altyapi" in t:                  return 2
    if "bağış" in t or "bagis" in t or "koordinasyon" in t: return 3
    if "diğer" in t or "diger" in t or "ilgisiz" in t:    return 4
    return -1   # hiçbiri eşleşmedi → parse başarısız

def zero_shot_siniflandir(tweet, model, tokenizer, max_new_tokens=12):
    prompt = build_prompt(tweet)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs['input_ids'].shape[1]

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    sure_ms = (time.time() - t0) * 1000   # GERÇEK süre (artık sabit 0 değil)

    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return response, sure_ms

# Tekil test
test_res, test_sure = zero_shot_siniflandir("Enkaz altında sesler var acil yardım.", model, tokenizer)
print(f"Örnek yanıt : {repr(test_res)}")
print(f"Tahmin ID   : {parse_label(test_res)}")
print(f"Süre        : {test_sure:.0f} ms")

Örnek yanıt : 'Tweet: Enkaz altında sesler var'
Tahmin ID   : -1
Süre        : 1973 ms


In [16]:
from tqdm import tqdm

print(f"Zero-shot başladı: {len(test_df)} tweet, model: TinyLlama-1.1B\n")

sonuclar = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    tweet  = row['tweet']
    gercek = row['label_id']

    yanit, sure = zero_shot_siniflandir(tweet, model, tokenizer)
    tahmin   = parse_label(yanit)

    sonuclar.append({
        'tweet'        : tweet,
        'gercek'       : gercek,
        'gercek_etiket': LABEL_NAMES[gercek],
        'model_yanit'  : yanit,
        'tahmin'       : tahmin,                              # -1 olabilir
        'tahmin_etiket': LABEL_NAMES.get(tahmin, "Parse Edilemedi"),
        'dogru_mu'     : (tahmin == gercek),
        'sure_ms'      : sure
    })

df_sonuc = pd.DataFrame(sonuclar)
print(f"\n✅ Tamamlandı!")
print(f"   Parse edilemeyen : {(df_sonuc['tahmin'] == -1).sum()} / {len(df_sonuc)}")
print(f"   Ortalama süre    : {df_sonuc['sure_ms'].mean():.1f} ms/tweet")

Zero-shot başladı: 80 tweet, model: TinyLlama-1.1B



100%|██████████| 80/80 [00:40<00:00,  1.97it/s]


✅ Tamamlandı!
   Parse edilemeyen : 79 / 80
   Ortalama süre    : 505.1 ms/tweet


In [18]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Parse edilemeyen (-1) → "Diğer/İlgisiz" (4) say: model geçerli kategori üretemedi demek
df_sonuc['tahmin_final'] = df_sonuc['tahmin'].replace(-1, 4)

y_true = df_sonuc['gercek'].tolist()
y_pred = df_sonuc['tahmin_final'].tolist()

accuracy    = accuracy_score(y_true, y_pred)
f1_macro    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("=" * 50)
print("TinyLlama-1.1B — ZERO-SHOT SONUÇLARI")
print("=" * 50)
print(f"Accuracy       : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"F1 Macro       : {f1_macro:.4f}")
print(f"F1 Weighted    : {f1_weighted:.4f}")
print(f"Inference hızı : {df_sonuc['sure_ms'].mean():.1f} ms/tweet")
print(f"Parse edilemedi: {(df_sonuc['tahmin'] == -1).sum()}/{len(df_sonuc)} (Diğer'e atandı)")
print()
print(classification_report(
    y_true, y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[LABEL_NAMES[i] for i in range(5)],
    zero_division=0))

TinyLlama-1.1B — ZERO-SHOT SONUÇLARI
Accuracy       : 0.2625  (26.2%)
F1 Macro       : 0.0941
F1 Weighted    : 0.1252
Inference hızı : 505.1 ms/tweet
Parse edilemedi: 79/80 (Diğer'e atandı)

                    precision    recall  f1-score   support

     Yardım Talebi       1.00      0.03      0.07        29
   Kayıp Bildirimi       0.00      0.00      0.00        10
    Altyapı Hasarı       0.00      0.00      0.00        10
Bağış/Koordinasyon       0.00      0.00      0.00        11
     Diğer/İlgisiz       0.25      1.00      0.40        20

          accuracy                           0.26        80
         macro avg       0.25      0.21      0.09        80
      weighted avg       0.43      0.26      0.13        80



In [19]:
import json, os

output_dir = '/content/drive/MyDrive/bdm_proje/results/zero_shot'
os.makedirs(output_dir, exist_ok=True)

# Model boyutunu güvenli hesapla (cell 7 çalışmasa bile)
model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

ozet = {
    "model"          : "TinyLlama-1.1B-Chat-v1.0",
    "model_id"       : "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "parametre"      : "1.1B",
    "yontem"         : "zero-shot",
    "test_size"      : int(len(test_df)),
    "gecerli_tahmin" : int(len(df_sonuc[df_sonuc['tahmin'] != -1])),
    "parse_edilemedi": int(len(df_sonuc[df_sonuc['tahmin'] == -1])),
    # >>> ASIL ÖNEMLİ METRİKLER — artık kaydediliyor <<<
    "accuracy"       : round(float(accuracy), 4),
    "f1_macro"       : round(float(f1_macro), 4),
    "f1_weighted"    : round(float(f1_weighted), 4),
    "inference_ms"   : round(float(df_sonuc['sure_ms'].mean()), 1),
    "model_boyutu_mb": round(float(model_size_mb), 0)
}

file_path = os.path.join(output_dir, 'tinyllama_zero_shot_ozet.json')
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)

print(f"✅ Kaydedildi: {file_path}")
print(json.dumps(ozet, ensure_ascii=False, indent=2))

✅ Kaydedildi: /content/drive/MyDrive/bdm_proje/results/zero_shot/tinyllama_zero_shot_ozet.json
{
  "model": "TinyLlama-1.1B-Chat-v1.0",
  "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "parametre": "1.1B",
  "yontem": "zero-shot",
  "test_size": 80,
  "gecerli_tahmin": 1,
  "parse_edilemedi": 79,
  "accuracy": 0.2625,
  "f1_macro": 0.0941,
  "f1_weighted": 0.1252,
  "inference_ms": 505.1,
  "model_boyutu_mb": 2200.0
}
